In [ ]:
!pip install -q decord

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install transformers

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


In [ ]:
import os
OUTPUT_PATH = "CLIP-features"
SEGMENT_PATH = "bjob-dataset/segments"
os.makedirs(OUTPUT_PATH, exist_ok = True)


In [ ]:
# Add this as a NEW CELL after cell 5 (after loading vision_model)
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize, InterpolationMode
from PIL import Image
from decord import VideoReader, cpu
import torch
import numpy as np

def load_video(video_path: str, size=224):
    def preprocess(size, n_px):
        return Compose([
            Resize(size, interpolation=InterpolationMode.BICUBIC),
            CenterCrop(size),
            lambda image: image.convert("RGB"),
            ToTensor(),
            Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
        ])(n_px)
    try:
        vr = VideoReader(video_path, num_threads=1, ctx=cpu(0))
        if len(vr) == 0:
            print(f"Warning: Video is empty or unreadable: {video_path}")
            return torch.empty(0)

        frames = vr.get_batch(range(len(vr))).asnumpy()

    except Exception as e:
        print(f"ERROR: Failed to read video file {video_path} with decord: {e}")
        return torch.empty(0)

    processed_frames_list = [
        preprocess(size, Image.fromarray(frame).convert("RGB"))
        for frame in frames
    ]

    if not processed_frames_list:
         print(f"Warning: No frames were processed from video: {video_path}")
         return torch.empty(0)

    video_frames_tensor = torch.stack(processed_frames_list)

    return video_frames_tensor

In [ ]:
from transformers import CLIPVisionModelWithProjection
model_name = "Searchium-ai/clip4clip-webvid150k"
vision_model = CLIPVisionModelWithProjection.from_pretrained(model_name).eval()
vision_model = vision_model.to(device)
print("Vision model moved to GPU")

def fetch_video_feature(video_path: str):
    video = load_video(video_path)
    
    if video.nelement() == 0:
        return video
    
    # Move video tensors to GPU
    video = video.to(device)
    
    with torch.no_grad():  # Save memory
        outputs = vision_model(video)
        outputs = outputs["image_embeds"]
        outputs = outputs / outputs.norm(dim=-1, keepdim=True)
        outputs = torch.mean(outputs, dim=0)
        outputs = outputs / outputs.norm(dim=-1, keepdim=True)
    
    return outputs
fetch_video_feature("segments/L26_V188/L26_V188_0_20.mp4").cpu().detach().numpy().shape

In [ ]:
from transformers import CLIPTokenizer, CLIPTextModelWithProjection
model_name = "Searchium-ai/clip4clip-webvid150k"
tokenizer = CLIPTokenizer.from_pretrained(model_name)
text_model = CLIPTextModelWithProjection.from_pretrained(model_name)
# Add this right after: text_model = CLIPTextModelWithProjection.from_pretrained(model_name)
text_model = text_model.to(device)
print("Text model moved to GPU")


def fetch_text_feature(text: str):
    inputs = tokenizer(text=text , return_tensors="pt")
    outputs = text_model(**inputs)

    outputs = outputs[0] / outputs[0].norm(dim=-1, keepdim=True)
    return outputs

fetch_text_feature("Vegetables on a white surface").cpu().detach().numpy().squeeze(0).shape

In [ ]:
test_vid_vec = fetch_video_feature("segments/L26_V188/L26_V188_0_20.mp4")
test_text_vec = (fetch_text_feature("Two people standing in a kitchen wearing white aprons").squeeze(0))

((test_vid_vec @ test_text_vec) * 0.5 + 0.5) * 100

In [ ]:
import os
import numpy as np

video_list = os.listdir(SEGMENT_PATH)

for video in video_list:
    video_folder_path = os.path.join(SEGMENT_PATH, video)

    if not os.path.isdir(video_folder_path):
        continue

    segments = os.listdir(video_folder_path)

    for segment in segments:
        if not segment.endswith('.mp4'):
            continue

        full_segment_path = os.path.join(video_folder_path, segment)

        if not os.path.isfile(full_segment_path):
            continue

        # ✅ Skip files that are 0 KB
        if os.path.getsize(full_segment_path) == 0:
            print(f"Skipping {full_segment_path} (0 KB file)")
            continue

        segment_base_name = os.path.splitext(segment)[0]  # e.g. L26_V188_0_20
        base_video_name = "_".join(segment_base_name.split("_")[:2])  # e.g. L26_V188

        # ✅ Create a subfolder for each base video name
        output_folder = os.path.join(OUTPUT_PATH, base_video_name)
        os.makedirs(output_folder, exist_ok=True)

        # Save path inside the video’s folder
        output_filename = f"{segment_base_name}.npy"
        file_path = os.path.join(output_folder, output_filename)

        if os.path.exists(file_path):
            print(f"Skipping {output_filename} (features already exist)")
            continue

        try:
            vid_feat = fetch_video_feature(full_segment_path)

            if vid_feat is None or vid_feat.nelement() == 0:
                print(f"Skipped {full_segment_path} (empty or failed to load)")
                continue

            np.save(file_path, vid_feat.cpu().detach().numpy())
            print(f"Saved {file_path}")

        except Exception as e:
            print(f"!! Error processing {full_segment_path}: {e}")


Saved CLIP-features/K07_V030/K07_V030_5961_6557.npy
Saved CLIP-features/K07_V030/K07_V030_20733_20912.npy
Saved CLIP-features/K07_V030/K07_V030_3385_3448.npy
Saved CLIP-features/K07_V030/K07_V030_2961_3052.npy
Saved CLIP-features/K07_V030/K07_V030_9561_9600.npy
Saved CLIP-features/K07_V030/K07_V030_5179_5244.npy
Saved CLIP-features/K07_V030/K07_V030_23107_23228.npy
Saved CLIP-features/K07_V030/K07_V030_24073_24140.npy
Saved CLIP-features/K07_V030/K07_V030_23489_23582.npy
Saved CLIP-features/K07_V030/K07_V030_21875_21917.npy
Saved CLIP-features/K07_V030/K07_V030_24243_24297.npy
Saved CLIP-features/K07_V030/K07_V030_9137_9180.npy
Saved CLIP-features/K07_V030/K07_V030_9462_9560.npy
Saved CLIP-features/K07_V030/K07_V030_14509_14717.npy
Saved CLIP-features/K07_V030/K07_V030_23373_23412.npy
Saved CLIP-features/K07_V030/K07_V030_6609_6669.npy
Saved CLIP-features/K07_V030/K07_V030_11983_12024.npy
Saved CLIP-features/K07_V030/K07_V030_20543_20639.npy
Saved CLIP-features/K07_V030/K07_V030_24705_